# OG-001: Private source transport and environment validation

This runner validates a pinned history-free source snapshot and the Kaggle T4 dependency overlay. It does not execute a scientific experiment. Original source identity, snapshot identity, and notebook commit are recorded separately.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import hashlib
import importlib.util
import json
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import traceback
import uuid
from datetime import datetime, timezone

SOURCE_ORIGIN_SHA = "91b6aa39e520f9312b098997134c1deabeabfb8f"
SOURCE_TREE_SHA = "1a6bd790404e39fae90f96b60cf9700a59bc2de5"
SOURCE_MANIFEST_SHA256 = "d64749565df66adc5be617aea959f2ff676eefbf19526dacdcc1d445f214046c"
DATASET_ID = "muhammadii/fq-ssiuu-source-91b6aa39"
# Staging fills this parameter with the commit containing this notebook.
ORCHESTRATION_COMMIT = "STAGING_REQUIRED"


def sha256(path):
    with open(path, "rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def write_json(path, data):
    with tempfile.NamedTemporaryFile(mode="w", dir=path.parent, delete=False) as handle:
        json.dump(data, handle, indent=2)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(handle.name, path)


def checked_git(cwd, *args):
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git {args[0]} failed, exit {result.returncode}")
    return result.stdout.strip()


def run_validation(input_root=Path("/kaggle/input"), working=Path("/kaggle/working")):
    working.mkdir(parents=True, exist_ok=True)
    repo = working / "fq-ssiuu"
    output = working / "og001_output"
    summary = {
        "schema_version": "2.0", "orchestration_run_id": str(uuid.uuid4()),
        "start_utc": datetime.now(timezone.utc).isoformat(),
        "source_origin_sha": SOURCE_ORIGIN_SHA, "source_tree_sha": SOURCE_TREE_SHA,
        "source_manifest_sha256": SOURCE_MANIFEST_SHA256,
        "orchestration_git_commit": ORCHESTRATION_COMMIT,
        "dataset_id": DATASET_ID, "source_validation_status": "NOT_REACHED",
        "setup_returncode": None, "execution_status": "FAIL",
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "scientific_experiment_executed": False,
    }
    phase = "source_validation"
    owned_repo = False
    try:
        if ORCHESTRATION_COMMIT == "STAGING_REQUIRED":
            raise RuntimeError("Notebook staging provenance has not been supplied")
        candidates = [p for p in input_root.rglob("SOURCE_MANIFEST.json")
                      if sha256(p) == SOURCE_MANIFEST_SHA256]
        if len(candidates) != 1:
            raise RuntimeError("Expected exactly one attached source manifest with the pinned SHA-256")
        manifest_path = candidates[0]
        manifest = json.loads(manifest_path.read_text())
        if (manifest.get("source_origin_sha") != SOURCE_ORIGIN_SHA
                or manifest.get("source_tree_sha") != SOURCE_TREE_SHA
                or manifest.get("snapshot_ref") != "kaggle-snapshot"
                or manifest.get("bundle_filename") != "fq-ssiuu-source.bundle"):
            raise RuntimeError("Source manifest provenance mismatch")
        bundle = manifest_path.parent / "fq-ssiuu-source.bundle"
        if (sha256(bundle) != manifest["bundle_sha256"]
                or bundle.stat().st_size != manifest["bundle_size_bytes"]):
            raise RuntimeError("Source bundle SHA-256 or size mismatch")
        if repo.exists() or output.exists():
            raise RuntimeError("Source checkout or setup output already exists")
        checked_git(working, "clone", "--no-checkout", "-b", "kaggle-snapshot", str(bundle), str(repo))
        owned_repo = True
        head = checked_git(repo, "rev-parse", "HEAD")
        tree = checked_git(repo, "rev-parse", "HEAD^{tree}")
        if head != manifest["snapshot_commit_sha"] or tree != SOURCE_TREE_SHA:
            raise RuntimeError("Cloned snapshot HEAD or tree mismatch")
        if checked_git(repo, "rev-list", "--count", "--all") != "1":
            raise RuntimeError("Source snapshot contains history")
        if len(checked_git(repo, "rev-list", "--parents", "-n", "1", "HEAD").split()) != 1:
            raise RuntimeError("Source snapshot has a parent")
        checked_git(repo, "checkout", "kaggle-snapshot")
        if checked_git(repo, "status", "--porcelain"):
            raise RuntimeError("Source snapshot is dirty")
        # Only load infrastructure code after its entire source tree is verified.
        spec = importlib.util.spec_from_file_location("source_transport", repo / "scripts/package_kaggle_source.py")
        transport = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(transport)
        transport.verify_bundle(manifest_path.parent, SOURCE_ORIGIN_SHA, SOURCE_MANIFEST_SHA256)
        summary.update({"source_validation_status": "PASS", "snapshot_commit_sha": head,
                        "bundle_sha256": manifest["bundle_sha256"], "history_commit_count": 1})
        write_json(working / "SOURCE_MANIFEST.json", manifest)
        phase = "setup"
        setup = subprocess.run([sys.executable, str(repo / "scripts/kaggle_setup.py"),
                                "--apply", "--output-dir", str(output)],
                               cwd=repo, capture_output=True, text=True)
        summary["setup_returncode"] = setup.returncode
        (working / "kaggle_runner_stdout.log").write_text(setup.stdout)
        (working / "kaggle_runner_stderr.log").write_text(setup.stderr)
        if setup.returncode:
            raise RuntimeError(f"Environment setup failed with exit {setup.returncode}; see setup logs")
        phase = "evidence_validation"
        required = ["setup_status.json", "MANIFEST.json", "before_manifest.json", "after_manifest.json",
                    "verifier_execution.json", "kaggle_env_verification.json", "overlay_lock.json",
                    "kaggle-overlay.lock", "pip_install_report.json", "pip_check.log"]
        for name in required:
            if not (output / name).is_file():
                raise RuntimeError(f"Missing setup evidence: {name}")
        records = {name: json.loads((output / name).read_text())
                   for name in required if name.endswith(".json")}
        run_id = records["MANIFEST.json"]["run_id"]
        summary["setup_run_id"] = run_id
        for name, sha_field in [("MANIFEST.json", "git_sha"), ("before_manifest.json", "git_commit"),
                                ("after_manifest.json", "git_commit"), ("verifier_execution.json", "setup_git_sha"),
                                ("kaggle_env_verification.json", "git_sha"), ("overlay_lock.json", "git_sha")]:
            if records[name].get("run_id") != run_id or records[name].get(sha_field) != head:
                raise RuntimeError(f"Run ID or snapshot linkage mismatch: {name}")
        status = records["setup_status.json"]
        if status.get("overall_status") != "PASS" or status.get("mode") != "APPLY":
            raise RuntimeError("Setup did not record APPLY PASS")
        for key in ("install_exit_code", "editable_exit_code", "pip_check_exit_code", "verifier_exit_code"):
            if status.get(key) != 0:
                raise RuntimeError(f"Setup return code is not zero: {key}")
        for name in ("before_manifest.json", "after_manifest.json"):
            if records[name].get("snapshot_status") != "PASS":
                raise RuntimeError(f"Environment snapshot failed: {name}")
        verifier = records["kaggle_env_verification.json"]
        required_checks = ("cuda_visible_devices_var", "cuda_device_count", "t4_hardware", "fp16_operation",
                           "sdpa_operation", "protected_stack_integrity", "overlay_imports",
                           "nf4_quantise_dequantise", "pip_check")
        if verifier.get("statuses", {}).get("overall") != "PASS":
            raise RuntimeError("Environment verifier did not pass")
        if any(verifier.get("checks", {}).get(key, {}).get("status") != "PASS" for key in required_checks):
            raise RuntimeError("Required environment check did not pass")
        execution = records["verifier_execution.json"]
        if (execution.get("verdict") != "PASS" or execution.get("verifier_returncode") != 0
                or execution.get("run_id_valid") is not True or execution.get("git_sha_valid") is not True):
            raise RuntimeError("Verifier execution linkage failed")
        for entry in records["MANIFEST.json"]["artifacts"]:
            path = (output / entry["path"]).resolve()
            if not path.is_relative_to(output.resolve()) or not path.is_file():
                raise RuntimeError("Invalid artifact manifest path")
            if path.stat().st_size != entry["size_bytes"] or sha256(path) != entry["sha256"]:
                raise RuntimeError("Artifact hash mismatch")
        summary["execution_status"] = "PASS"
    except Exception as exc:
        summary["failure_phase"] = phase
        summary["first_failure"] = f"{type(exc).__name__}: {exc}"
        (working / "orchestration_failure.log").write_text(traceback.format_exc())
    finally:
        # The generated checkout is disposable; all run evidence is outside it.
        if owned_repo:
            try:
                shutil.rmtree(repo)
            except OSError as exc:
                summary["cleanup_error"] = type(exc).__name__
        summary["end_utc"] = datetime.now(timezone.utc).isoformat()
        summary["setup_status_file_exists"] = (output / "setup_status.json").is_file()
        write_json(working / "kernel_summary.json", summary)
        print(json.dumps(summary, indent=2))
    if summary["execution_status"] != "PASS":
        raise RuntimeError(summary.get("first_failure", "Validation failed"))
    return summary


if __name__ == "__main__":
    run_validation()
